In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# NLP
import re
import string
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

# ML
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

In [2]:
df = pd.read_csv("Starbucks satisfactory survey.csv")

df.head()

,Timestamp,1. Your Gender,2. Your Age,3. Are you currently....?,4. What is your annual income?,5. How often do you visit Starbucks?,6. How do you usually enjoy Starbucks?,7. How much time do you normally spend during your visit?,8. The nearest Starbucks's outlet to you is...?,9. Do you have Starbucks membership card?,10. What do you most frequently purchase at Starbucks?,"11. On average, how much would you spend at Starbucks per visit?","12. How would you rate the quality of Starbucks compared to other brands (Coffee Bean, Old Town White Coffee..) to be:",13. How would you rate the price range at Starbucks?,14. How important are sales and promotions in your purchase decision?,"15. How would you rate the ambiance at Starbucks? (lighting, music, etc...)",16. You rate the WiFi quality at Starbucks as..,"17. How would you rate the service at Starbucks? (Promptness, friendliness, etc..)",18. How likely you will choose Starbucks for doing business meetings or hangout with friends?,19. How do you come to hear of promotions at Starbucks? Check all that apply.,20. Will you continue buying at Starbucks?
0,2019/10/01 12:38:43 PM GMT+8,Female,From 20 to 29,Student,"Less than RM25,000",Rarely,Dine in,Between 30 minutes to 1 hour,within 1km,Yes,Coffee,Less than RM20,4,3,5,5,4,4,3,Starbucks Website/Apps;Social Media;Emails;Dea...,Yes
1,2019/10/01 12:38:54 PM GMT+8,Female,From 20 to 29,Student,"Less than RM25,000",Rarely,Take away,Below 30 minutes,1km - 3km,Yes,Cold drinks;Pastries,Less than RM20,4,3,4,4,4,5,2,Social Media;In Store displays,Yes
2,2019/10/01 12:38:56 PM GMT+8,Male,From 20 to 29,Employed,"Less than RM25,000",Monthly,Dine in,Between 30 minutes to 1 hour,more than 3km,Yes,Coffee,Less than RM20,4,3,4,4,4,4,3,In Store displays;Billboards,Yes
3,2019/10/01 12:39:08 PM GMT+8,Female,From 20 to 29,Student,"Less than RM25,000",Rarely,Take away,Below 30 minutes,more than 3km,No,Coffee,Less than RM20,2,1,4,3,3,3,3,Through friends and word of mouth,No
4,2019/10/01 12:39:20 PM GMT+8,Male,From 20 to 29,Student,"Less than RM25,000",Monthly,Take away,Between 30 minutes to 1 hour,1km - 3km,No,Coffee;Sandwiches,Around RM20 - RM40,3,3,4,2,2,3,3,Starbucks Website/Apps;Social Media,Yes


In [3]:
print(df.info())
print(df.isnull().sum())

df.describe(include="all")

<class 'pandas.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 21 columns):
 #   Column                                                                                                                  Non-Null Count  Dtype
---  ------                                                                                                                  --------------  -----
 0   Timestamp                                                                                                               122 non-null    str  
 1   1. Your Gender                                                                                                          122 non-null    str  
 2   2. Your Age                                                                                                             122 non-null    str  
 3   3. Are you currently....?                                                                                               122 non-null    str  
 4   4. What is your ann

,Timestamp,1. Your Gender,2. Your Age,3. Are you currently....?,4. What is your annual income?,5. How often do you visit Starbucks?,6. How do you usually enjoy Starbucks?,7. How much time do you normally spend during your visit?,8. The nearest Starbucks's outlet to you is...?,9. Do you have Starbucks membership card?,10. What do you most frequently purchase at Starbucks?,"11. On average, how much would you spend at Starbucks per visit?","12. How would you rate the quality of Starbucks compared to other brands (Coffee Bean, Old Town White Coffee..) to be:",13. How would you rate the price range at Starbucks?,14. How important are sales and promotions in your purchase decision?,"15. How would you rate the ambiance at Starbucks? (lighting, music, etc...)",16. You rate the WiFi quality at Starbucks as..,"17. How would you rate the service at Starbucks? (Promptness, friendliness, etc..)",18. How likely you will choose Starbucks for doing business meetings or hangout with friends?,19. How do you come to hear of promotions at Starbucks? Check all that apply.,20. Will you continue buying at Starbucks?
count,122,122,122,122,122,122,121,122,122,122,122,122,122.000000,122.000000,122.000000,122.000000,122.000000,122.000000,122.000000,121,122
unique,122,2,4,4,5,5,8,5,3,2,20,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31,2
top,2019/10/01 12:38:43 PM GMT+8,Female,From 20 to 29,Employed,"Less than RM25,000",Rarely,Take away,Below 30 minutes,more than 3km,No,Coffee,Less than RM20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Social Media,Yes
freq,1,65,85,61,71,76,49,73,61,62,65,58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31,94
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.663934,2.893443,3.795082,3.754098,3.254098,3.745902,3.516393,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.941343,1.081836,1.090443,0.929867,0.958317,0.828834,1.030394,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.000000,2.000000,3.000000,3.000000,3.000000,3.000000,3.000000,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.000000,3.000000,4.000000,4.000000,3.000000,4.000000,4.000000,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.000000,4.000000,5.000000,4.000000,4.000000,4.000000,4.000000,NaN,NaN


In [5]:
print(df.columns.tolist())

['Timestamp', '1. Your Gender', '2. Your Age', '3. Are you currently....?', '4. What is your annual income?', '5. How often do you visit Starbucks?', '6. How do you usually enjoy Starbucks?', '7. How much time do you normally  spend during your visit?', "8. The nearest Starbucks's outlet to you is...?", '9. Do you have Starbucks membership card?', '10. What do you most frequently purchase at Starbucks?', '11. On average, how much would you spend at Starbucks per visit?', '12. How would you rate the quality of Starbucks compared to other brands (Coffee Bean, Old Town White Coffee..) to be:', '13. How would you rate the price range at Starbucks?', '14. How important are sales and promotions in your purchase decision?', '15. How would you rate the ambiance at Starbucks? (lighting, music, etc...)', '16. You rate the WiFi quality at Starbucks as..', '17. How would you rate the service at Starbucks? (Promptness, friendliness, etc..)', '18. How likely you will choose Starbucks for doing busin

In [7]:
print(df.columns.tolist())

['Timestamp', '1. Your Gender', '2. Your Age', '3. Are you currently....?', '4. What is your annual income?', '5. How often do you visit Starbucks?', '6. How do you usually enjoy Starbucks?', '7. How much time do you normally  spend during your visit?', "8. The nearest Starbucks's outlet to you is...?", '9. Do you have Starbucks membership card?', '10. What do you most frequently purchase at Starbucks?', '11. On average, how much would you spend at Starbucks per visit?', '12. How would you rate the quality of Starbucks compared to other brands (Coffee Bean, Old Town White Coffee..) to be:', '13. How would you rate the price range at Starbucks?', '14. How important are sales and promotions in your purchase decision?', '15. How would you rate the ambiance at Starbucks? (lighting, music, etc...)', '16. You rate the WiFi quality at Starbucks as..', '17. How would you rate the service at Starbucks? (Promptness, friendliness, etc..)', '18. How likely you will choose Starbucks for doing busin

In [10]:
print(df.columns)


Index(['Timestamp', '1. Your Gender', '2. Your Age',
       '3. Are you currently....?', '4. What is your annual income?',
       '5. How often do you visit Starbucks?',
       '6. How do you usually enjoy Starbucks?',
       '7. How much time do you normally  spend during your visit?',
       '8. The nearest Starbucks's outlet to you is...?',
       '9. Do you have Starbucks membership card?',
       '10. What do you most frequently purchase at Starbucks?',
       '11. On average, how much would you spend at Starbucks per visit?',
       '12. How would you rate the quality of Starbucks compared to other brands (Coffee Bean, Old Town White Coffee..) to be:',
       '13. How would you rate the price range at Starbucks?',
       '14. How important are sales and promotions in your purchase decision?',
       '15. How would you rate the ambiance at Starbucks? (lighting, music, etc...)',
       '16. You rate the WiFi quality at Starbucks as..',
       '17. How would you rate the service at 

In [12]:
print(df.columns)

Index(['Timestamp', '1. Your Gender', '2. Your Age',
       '3. Are you currently....?', '4. What is your annual income?',
       '5. How often do you visit Starbucks?',
       '6. How do you usually enjoy Starbucks?',
       '7. How much time do you normally  spend during your visit?',
       '8. The nearest Starbucks's outlet to you is...?',
       '9. Do you have Starbucks membership card?',
       '10. What do you most frequently purchase at Starbucks?',
       '11. On average, how much would you spend at Starbucks per visit?',
       '12. How would you rate the quality of Starbucks compared to other brands (Coffee Bean, Old Town White Coffee..) to be:',
       '13. How would you rate the price range at Starbucks?',
       '14. How important are sales and promotions in your purchase decision?',
       '15. How would you rate the ambiance at Starbucks? (lighting, music, etc...)',
       '16. You rate the WiFi quality at Starbucks as..',
       '17. How would you rate the service at 

In [14]:
print(df.columns.tolist())
print("X exists:", "X" in globals())

['Timestamp', '1. Your Gender', '2. Your Age', '3. Are you currently....?', '4. What is your annual income?', '5. How often do you visit Starbucks?', '6. How do you usually enjoy Starbucks?', '7. How much time do you normally  spend during your visit?', "8. The nearest Starbucks's outlet to you is...?", '9. Do you have Starbucks membership card?', '10. What do you most frequently purchase at Starbucks?', '11. On average, how much would you spend at Starbucks per visit?', '12. How would you rate the quality of Starbucks compared to other brands (Coffee Bean, Old Town White Coffee..) to be:', '13. How would you rate the price range at Starbucks?', '14. How important are sales and promotions in your purchase decision?', '15. How would you rate the ambiance at Starbucks? (lighting, music, etc...)', '16. You rate the WiFi quality at Starbucks as..', '17. How would you rate the service at Starbucks? (Promptness, friendliness, etc..)', '18. How likely you will choose Starbucks for doing busin